# Customers Like You: ML analysis

ML Engineer notebook (tasks ML2 to ML7). Run it top to bottom from a fresh kernel.
Every decision is logged in `outputs/ml_decisions.md` (R1); the method choice (ML1) was written before any fitting.

| Section | Task | Output files |
|---|---|---|
| 0 | Load and check the data | none |
| 1 | ML2: k-sweep | `ml2_k_sweep.csv`, `ml2_k_sweep.png` |
| 2 | ML3: recommendations | `ml3_similar_examples.csv` |
| 3 | ML4: hit rate vs baseline, nightly job | `ml4_hit_rates.csv`, `ml4_hit_rate_by_k.csv`, `ml4_nightly_job.csv` |
| 4 | ML5: PCA vs Truncated SVD | `ml5_comparison.csv` |
| 5 | ML6: customer map | `ml6_cluster_choice.csv`, `ml6_customer_map.png`, `ml6_segments.csv` |
| 6 | ML7: sensitivity to the purchase value | `ml7_sensitivity.csv` |

In [ ]:
import os, sys
from pathlib import Path

# make the notebook work whether VS Code starts it in notebooks/ or in the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.ml_data import load_all, hit_rate_all_customers, change_cell_value
from src.models import (SEED, fit_svd, fit_pca, sweep_k, plot_k_sweep,
                        volume_correlation, top_products_for_component)
from src.recommend import product_vectors, similar_products, popularity, recommend_for_customers
from src.segments import map_coordinates, cluster_customers, relabel_by_volume, profile_segments
from src.evaluate import evaluate_recommender, time_nightly_job

OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
np.random.seed(SEED)          # R6: fixed seed (every fit also gets random_state=SEED)
print("repo root:", ROOT.name, "| seed:", SEED)

## 0. Load and check the data

The Data Engineer's matrix and test pairs. Two checks before anything is fitted:
the matrix must hold only purchases (all values positive), and no held-out purchase may appear in it (R2).

In [ ]:
d = load_all()
A = d.A
print(f"matrix: {A.shape[0]:,} customers x {A.shape[1]:,} products, {A.nnz:,} non-zeros, "
      f"sparsity {1 - A.nnz / (A.shape[0] * A.shape[1]):.2%}")
print("test customers:", d.n_test_customers, "| scorable:", len(d.holdout))
print("test pairs used:", d.holdout_stats["test_pairs_used"], "of", d.holdout_stats["test_pairs"])

assert A.data.min() > 0, "matrix has non-positive values"
leaks = sum((np.asarray(A[np.full(len(c), r), c]).ravel() != 0).sum() for r, c in d.holdout.items())
assert leaks == 0, f"{leaks} held-out purchases are inside the training matrix"
print("checks passed: all values positive, no held-out purchase in the training matrix")

## 1. ML2: how many components (k)?

Truncated SVD (the main method, see ML1) is fitted for k = 10, 20, 50, 100, 200 on the training matrix.
For each k we record:

- **relative reconstruction error**: how much of the purchase data is lost by compressing (lower is better)
- **fit time**, **peak memory** and **size of the stored factors**: what each k costs

As set in ML1, k is chosen from this curve and the costs, **not** from the test hit rate.

In [ ]:
KS = (10, 20, 50, 100, 200)
svd_sweep = sweep_k(A, ks=KS, method="svd")
svd_sweep.to_csv(OUT / "ml2_k_sweep.csv", index=False)
display(svd_sweep)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
plot_k_sweep({"svd": svd_sweep}, ax=ax)
ax.set_xscale("log")
ax.set_xticks(list(KS)); ax.set_xticklabels(KS)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "ml2_k_sweep.png", dpi=150)
plt.show()

# how much each extra 10 components buys, between neighbouring k values
gain = svd_sweep[["k"]].copy()
gain["error_drop_per_10_extra_components"] = (
    -svd_sweep["rel_error"].diff() / svd_sweep["k"].diff() * 10).round(4)
gain["stored_mb"] = svd_sweep["factors_mb"]
display(gain)

### Choosing k

- The error falls steadily with no clear elbow (0.88 at k=10, 0.64 at k=200). The gain per extra 10 components shrinks but never reaches zero.
- Cost does not decide it: even k=200 fits in about 3 seconds and stores 14.6 MB.
- k=50 sits mid-curve. It fits in under a second and stores about 3.6 MB, and it keeps broad buying patterns rather than individual quirks.
- **Chosen k: 50.** Chosen from the curve and the costs only. The test hit rate was not used (ML1).

In [ ]:
CHOSEN_K = 50   # <-- YOUR decision from ML2. Edit here if you choose differently, and log the reason.
assert CHOSEN_K is not None, "set CHOSEN_K above"
print("CHOSEN_K =", CHOSEN_K)

## 2. ML3: recommendations

Fit Truncated SVD at the chosen k on the training matrix. Each product becomes a short vector (its row of the
product factors, scaled to length 1). Then:

- **Similar products** ("customers who bought this also bought"): the 10 products whose vectors have the highest cosine similarity.
- **Recommendations for a customer**: score every product by adding up its similarity to each product the customer bought, exclude what they already bought, and take the top 10.

In [ ]:
svd_fit = fit_svd(A, CHOSEN_K)
P = product_vectors(svd_fit.product_factors)       # one unit-length row per product
print(f"SVD fitted: k={CHOSEN_K}, fit time {svd_fit.fit_time:.2f}s, relative error {svd_fit.rel_error:.3f}")

In [ ]:
# "Customers who bought this also bought ..." for three example products:
# the best seller, plus two mid-popularity products picked with the fixed seed.
pop = popularity(A)
mid = np.where((pop >= np.quantile(pop, 0.5)) & (pop <= np.quantile(pop, 0.9)))[0]
rng = np.random.default_rng(SEED)
examples = [int(np.argmax(pop)), *map(int, rng.choice(mid, size=2, replace=False))]

tables = []
for j in examples:
    idx, sims = similar_products(P, j, n=10)
    t = pd.DataFrame({"query_product": d.names[j], "rank": range(1, 11),
                      "similar_product": [d.names[i] for i in idx], "cosine": np.round(sims, 3)})
    tables.append(t)
    print(f"Customers who bought '{d.names[j]}' also bought:")
    display(t[["rank", "similar_product", "cosine"]].set_index("rank"))
pd.concat(tables).to_csv(OUT / "ml3_similar_examples.csv", index=False)

In [ ]:
# Recommendations for one customer, shown by anonymised matrix row (never a Customer ID, R4)
row = int(sorted(d.holdout)[0])
bought = A[row].toarray().ravel()
top_bought = np.argsort(-bought)[:5]
recs = recommend_for_customers(A, P, [row], n=10)[0]

print(f"Customer #{row}: bought {int((bought > 0).sum())} products. Their 5 biggest purchases:")
display(pd.DataFrame({"product": [d.names[j] for j in top_bought]}))
print("Top 10 recommendations (products they have not bought):")
display(pd.DataFrame({"product": [d.names[j] for j in recs]}))

## 3. ML4: does it beat the best-seller baseline?

**Hit rate at 10**: the share of test customers with at least one held-out product in their top 10.
The held-out products are ones the customer never bought before the cutoff, so this tests *discovery*, not repeat buying.
The baseline shows the 10 best sellers (most distinct buyers) that the customer has not already bought (R3).

Two versions are reported (see `ml_decisions.md`):
- **all test customers** (headline): customers the model cannot score count as misses
- **scorable only**: customers who are in the matrix and have at least one held-out product that exists in it

In [ ]:
res = evaluate_recommender(A, P, d.holdout, n=10)
N, T = res["n_customers"], d.n_test_customers
scale = N / T                                       # scorable -> all test customers

hit_rates = pd.DataFrame([
    {"method": "SVD recommender",      "hit_rate_all_test_customers": res["model_hit_rate@10"] * scale,
     "std_error_all": res["model_se"] * scale,     "hit_rate_scorable_only": res["model_hit_rate@10"],
     "std_error_scorable": res["model_se"]},
    {"method": "Best-seller baseline", "hit_rate_all_test_customers": res["baseline_hit_rate@10"] * scale,
     "std_error_all": res["baseline_se"] * scale,  "hit_rate_scorable_only": res["baseline_hit_rate@10"],
     "std_error_scorable": res["baseline_se"]},
])
hit_rates["k"] = CHOSEN_K
hit_rates["customers_all"], hit_rates["customers_scorable"] = T, N
hit_rates.round(4).to_csv(OUT / "ml4_hit_rates.csv", index=False)
display(hit_rates.round(4))
print(f"Lift over baseline: {res['lift']:.2f}x   (customers: {N:,} scorable of {T:,})")

### What the numbers say

**The model did not beat the best-seller baseline.**

| | Hit rate at 10 (all 1,931 test customers) |
|---|---|
| SVD recommender (k=50) | 20.9% (±0.9) |
| Best-seller baseline | 26.8% (±1.0) |

- The lift is 0.78: the model finds a held-out purchase for about 22% fewer customers than the best sellers do.
- The gap is 5.9 points, about 4 standard errors, so it is not chance.
- It holds at every k tested: the model scores 14% to 22%, the baseline 27% (scorable customers).
- Using only the 1,916 scorable customers tells the same story: 21.1% against 27.0%.

**Possible reasons (not yet tested):**
- Best sellers are hard to beat in retail. New purchases are often popular products, and the baseline is pure popularity. Our scoring scales every product vector to the same length, so most of the popularity signal is lost.
- The test period (9 Oct to 9 Dec) is the pre-Christmas season, when many shops may buy seasonal best sellers.
- The test counts only products a customer has never bought, so it ignores reorders, which are a large part of this business. It is a hard test for the "also bought" idea.

**What this means for Sarah:** on this test, the suggestions are not better than showing best sellers. The nightly job is fast (0.7 seconds in total), so speed is not the problem. Accuracy is.

**What we did and did not do:** k stayed at 50 and the model was not changed after seeing the test result (R2). Next step: check the reasons above with diagnostics (overlap with the best sellers, and hit rate by customer activity).

In [ ]:
# Hit rate at the other k values: for information only. k was chosen in ML2 and is NOT changed by this table (ML1).
rows = []
for k in KS:
    f = fit_svd(A, k)
    r = evaluate_recommender(A, product_vectors(f.product_factors), d.holdout, n=10)
    rows.append({"k": k, "model_hit_rate_scorable": r["model_hit_rate@10"],
                 "baseline_hit_rate_scorable": r["baseline_hit_rate@10"], "lift": r["lift"]})
by_k = pd.DataFrame(rows)
by_k.round(4).to_csv(OUT / "ml4_hit_rate_by_k.csv", index=False)
display(by_k.round(4))

### Nightly job timing

The nightly job is: fit the model, build the "also bought" list for every product, and score every customer.

In [ ]:
timing = time_nightly_job(A, P, n=10)
job = {"fit_s": round(svd_fit.fit_time, 3), **timing}
job["total_s"] = round(sum(job.values()), 3)
job = pd.DataFrame([job])
job.to_csv(OUT / "ml4_nightly_job.csv", index=False)
display(job)

## 4. ML5: PCA vs Truncated SVD

The other method, run on the **same matrix, the same k and the same recommender** as the SVD above.
The only difference is centring: PCA subtracts each product's average first (done implicitly, so the matrix stays sparse).

Two notes for reading the table:
- PCA also has to store the column means (a small vector). That is included in its stored size.
- PCA's error is measured against the raw matrix, like SVD's, so the two error figures are comparable.

In [ ]:
pca_fit = fit_pca(A, CHOSEN_K)
P_pca = product_vectors(pca_fit.product_factors)
res_svd = res                                                   # SVD result from section 3
res_pca = evaluate_recommender(A, P_pca, d.holdout, n=10)
print(f"PCA fitted: k={CHOSEN_K}, fit time {pca_fit.fit_time:.2f}s, "
      f"relative error {pca_fit.rel_error:.3f}, explained variance {pca_fit.evr:.3f}")

### What is component 1?

Component 1 is the strongest pattern in the data. For each method we check whether it is simply
**how much a customer buys** (correlation with total purchases, close to 1 means yes) and look at the products that load on it.

In [ ]:
corr_svd = abs(volume_correlation(svd_fit, A))
corr_pca = abs(volume_correlation(pca_fit, A))
print(f"Correlation of component 1 with total purchases:  SVD {corr_svd:.2f}   PCA {corr_pca:.2f}")
print("\nSVD: products with the largest component-1 loadings")
display(top_products_for_component(svd_fit, d.names, 0, 8))
print("PCA: products with the largest component-1 loadings (the sign of a component is arbitrary)")
display(top_products_for_component(pca_fit, d.names, 0, 8))

In [ ]:
# Draft wording from the correlation. READ THE PRODUCT LISTS ABOVE, THEN EDIT THESE TWO LINES in your own words.
def describe(corr):
    if corr >= 0.7:
        return f"Mostly overall buying volume (correlation {corr:.2f} with how much a customer buys)"
    return f"Not mainly volume (correlation {corr:.2f} with how much a customer buys); see its top products"
COMP1_SVD, COMP1_PCA = describe(corr_svd), describe(corr_pca)
print("SVD:", COMP1_SVD, "\nPCA:", COMP1_PCA)

### Comparison table

In [ ]:
N, T = res_svd["n_customers"], d.n_test_customers
scale = N / T                                                    # scorable -> all test customers
pca_stored_mb = pca_fit.factors_mb + pca_fit.model.mean_.nbytes / 1024**2

comparison = pd.DataFrame({
    "Measure": [
        "k used",
        "Fit time (seconds)",
        "Peak memory during fit (MB, approximate)",
        "Size of stored factors (MB)",
        "Relative reconstruction error (vs raw matrix)",
        "Share of the matrix's squared size kept",
        "Variance explained (PCA only)",
        "Hit rate at 10, all test customers",
        "Hit rate at 10, scorable customers only",
        "Best-seller baseline hit rate, all test customers",
        "Best-seller baseline hit rate, scorable only",
        "Correlation of component 1 with total purchases",
        "What component 1 represents",
    ],
    "PCA": [
        CHOSEN_K, round(pca_fit.fit_time, 2), round(pca_fit.peak_mem_mb, 1), round(pca_stored_mb, 2),
        round(pca_fit.rel_error, 4), round(pca_fit.energy_kept, 4), round(pca_fit.evr, 4),
        round(res_pca["model_hit_rate@10"] * scale, 4), round(res_pca["model_hit_rate@10"], 4),
        round(res_pca["baseline_hit_rate@10"] * scale, 4), round(res_pca["baseline_hit_rate@10"], 4),
        round(corr_pca, 2), COMP1_PCA,
    ],
    "Truncated SVD": [
        CHOSEN_K, round(svd_fit.fit_time, 2), round(svd_fit.peak_mem_mb, 1), round(svd_fit.factors_mb, 2),
        round(svd_fit.rel_error, 4), round(svd_fit.energy_kept, 4), "n/a (data is not centred)",
        round(res_svd["model_hit_rate@10"] * scale, 4), round(res_svd["model_hit_rate@10"], 4),
        round(res_svd["baseline_hit_rate@10"] * scale, 4), round(res_svd["baseline_hit_rate@10"], 4),
        round(corr_svd, 2), COMP1_SVD,
    ],
})
comparison.to_csv(OUT / "ml5_comparison.csv", index=False)
with pd.option_context("display.max_colwidth", None):
    display(comparison.set_index("Measure"))
print(f"Lift over baseline (all customers):  PCA {res_pca['lift']:.2f}x   SVD {res_svd['lift']:.2f}x")

### What the comparison says

**SVD and PCA came out essentially the same.**

| | PCA | Truncated SVD |
|---|---|---|
| Hit rate at 10 (all test customers) | 20.2% | 20.9% |
| Relative reconstruction error | 0.789 | 0.790 |
| Fit time | 0.21 s | 0.28 s |
| Peak memory (approximate) | 7.0 MB | 6.8 MB |
| Stored factors | 3.68 MB | 3.64 MB |

The three predictions from ML1:
1. **Component 1 is buying volume: confirmed for both** (correlation 0.97). Even PCA, which removes the average customer first, is dominated by volume, because customers differ mostly in how much they buy.
2. **Similar hit rates: confirmed.** The 0.7-point gap is smaller than the standard error of each rate (about 0.9 points), so the two methods cannot be told apart. Both are below the best-seller baseline (lift 0.75 and 0.78).
3. **PCA costs less than the textbook suggests: confirmed.** PCA used about the same time and memory as SVD, because scikit-learn centres the sparse matrix implicitly instead of building a dense copy. Fit times under a second are noisy, so the small differences mean nothing.

The two "variance" figures are not comparable: PCA's 34.9% is variance around the average customer, SVD's 37.6% is the share of the matrix's squared size. The relative reconstruction error (same for both) is the fair comparison.

**Choice for the nightly job: Truncated SVD.** Not because it measured better (it did not, clearly), but because nothing favours PCA: SVD is simpler (no centring step), gives both sets of factors in one fit, and is the standard choice for this kind of data.

## 5. ML6: customer map

Marketing wants a simple picture of the main customer groups.
Each customer is placed on a plane using the first two SVD components, and k-means groups the customers.
Component 1 is buying volume (ML5), so the horizontal axis reads roughly "small buyer to big buyer".

The number of groups (3 to 6) is chosen by the **silhouette score** (higher means better separated groups),
set before looking at the groups. Each axis is standardised first, so volume does not decide everything.

In [ ]:
Z = map_coordinates(svd_fit.customer_factors, A)           # 2 columns; higher component 1 = bigger buyer
choice, all_labels = cluster_customers(Z, ks=(3, 4, 5, 6), seed=SEED)
choice.to_csv(OUT / "ml6_cluster_choice.csv", index=False)
display(choice)

In [ ]:
N_SEGMENTS_OVERRIDE = None        # leave as None to use the best silhouette; if you override, log the reason
N_SEGMENTS = N_SEGMENTS_OVERRIDE or int(choice.loc[choice["silhouette"].idxmax(), "n_segments"])
labels = relabel_by_volume(all_labels[N_SEGMENTS], Z)       # segment 0 = smallest buyers, increasing
print("number of segments:", N_SEGMENTS)

profile = profile_segments(A, labels, d.names, d.country)
overall_countries = pd.Series(d.country).value_counts(normalize=True).head(3)
print("country mix of all customers:", "; ".join(f"{c} {v:.0%}" for c, v in overall_countries.items()))
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(profile)

### Naming the segments

Each segment is named from three pieces of evidence in the profile table: how many different products its customers typically buy (the volume), the products it buys much more often than other customers (the theme), and its main countries. The light buyers are 83% of customers and have no distinctive products, so they are named by volume alone.

In [ ]:
SEGMENT_NAMES = {                                   # named from the profile table (see 'What the map says')
    0: "Light buyers",
    1: "Heavy buyers: home décor",
    2: "Heavy buyers: party and bags",
}
assert len(SEGMENT_NAMES) == N_SEGMENTS, 'update SEGMENT_NAMES: the number of segments changed'

names_per_customer = np.array([SEGMENT_NAMES[s] for s in labels])
sizes = pd.Series(labels).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5.5))
for s in range(N_SEGMENTS):
    m = labels == s
    ax.scatter(Z[m, 0], Z[m, 1], s=8, alpha=0.5, label=f"{SEGMENT_NAMES[s]} ({sizes[s]:,})")
if Z[:, 0].min() > 0:
    ax.set_xscale("log")
ax.set_xlabel("Component 1: overall buying volume (higher = bigger buyer)")
ax.set_ylabel("Component 2")
ax.set_title("Customer map: main customer groups")
ax.grid(alpha=0.3)
ax.legend(markerscale=2, fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "ml6_customer_map.png", dpi=150)
plt.show()

segments = profile.copy()
segments.insert(1, "name", segments["segment"].map(SEGMENT_NAMES))
segments.to_csv(OUT / "ml6_segments.csv", index=False)          # no customer IDs anywhere (R4)

### What the map says

**Three groups were chosen.** The silhouette score is highest at 3 groups (0.64). With 4 or more groups the score falls, because the extra groups are tiny (under 0.3% of customers, about 10 to 12 people), too small for a campaign.

| Group | Customers | Products bought (typical) | Main countries | What stands out |
|---|---|---|---|---|
| Light buyers | 4,542 (83%) | 32 | UK 92% | No product theme |
| Heavy buyers: home décor | 537 (10%) | 182 | UK 98% | Heart T-light holders (79% vs 26% overall), wicker hearts, picture frames |
| Heavy buyers: party and bags | 392 (7%) | 171 | UK 71%, Germany 9%, France 6% | Cake cases (68% vs 19%), lunch bags, jumbo shopper bag |

- **Part of this is a volume story.** The light buyers differ from the other two groups mainly in how much they buy (component 1).
- **It is not only volume.** The two heavy groups buy about the same amount (around 180 and 170 products) and differ by product theme (component 2).
- **Light buyers have no distinctive products** (the "0% vs 0%" in the first run). They are 83% of customers, so their product shares are almost the same as the overall shares. They are best described by what they do not buy: the themes the two heavy groups love.
- **Ideas for marketing** (examples, not tested): promote new hearts, wicker and frame lines to the first heavy group; promote lunch bags and paper goods to the second, which is also the most international (Germany and France); try to move light buyers into one of the two themes.
- **Limit:** the light-buyer group is too large and mixed for a single campaign. A next step could be to split it by product mix instead of volume.

## 6. ML7: does the purchase value change the hit rate?

The matrix cell can hold three different things:
- **log**: log(1 + total quantity), as delivered by the Data Engineer (used in all the results above)
- **count**: the total quantity, recovered exactly from the saved matrix
- **binary**: 1 if the customer ever bought the product, else 0

Each version is fitted with the same method (SVD), the same k and the same seed, and scored on the same held-out purchases with the same recommender.
This is a sensitivity check: the main matrix stays as delivered, and the matrix is frozen (Day 4), so a different value would not replace it here.

In [ ]:
rows = []
for mode in ("log", "count", "binary"):
    A_mode = change_cell_value(A, mode)
    f = fit_svd(A_mode, CHOSEN_K)
    r = evaluate_recommender(A_mode, product_vectors(f.product_factors), d.holdout, n=10)
    rows.append({
        "purchase_value": mode,
        "used_in_main_model": mode == "log",
        "hit_rate_all_test_customers": r["model_hit_rate@10"] * scale,
        "std_error_all": r["model_se"] * scale,
        "hit_rate_scorable_only": r["model_hit_rate@10"],
        "baseline_all_test_customers": r["baseline_hit_rate@10"] * scale,
        "lift_over_baseline": r["lift"],
        "relative_error": f.rel_error,
    })
sensitivity = pd.DataFrame(rows)
sensitivity.round(4).to_csv(OUT / "ml7_sensitivity.csv", index=False)
display(sensitivity.round(4))

### What the sensitivity check says

**The purchase value matters, and less quantity information gave better recommendations.**

| Purchase value | Hit rate at 10 (all 1,931 test customers) | Lift over baseline (26.8%) |
|---|---|---|
| log (main model) | 20.9% (±0.9) | 0.78 |
| count | 12.6% (±0.8) | 0.47 |
| binary (1/0) | 24.8% (±1.0) | 0.92 |

- **Ranking: binary > log > count.** Binary beats log by about 3.8 points, nearly 3 standard errors, so the gap is probably real. Count is far worse than both, about 8 points below log.
- **Why raw counts hurt:** a few very large orders dominate the matrix, so the model spends its components on those big buyers. The low error for count (0.19) shows this. It is low because a few huge cells dominate, not because the recommendations are good. The errors are not comparable across rows.
- **Why 1/0 works best:** quantity mostly reflects the size of a shop (see the Data Engineer's notes), so *who bought what* is a better signal than *how much* for finding companion products.
- **No version clearly beats the best-seller baseline.** Binary comes closest (24.8% vs 26.8%). The gap is about 2 points, roughly 1.4 standard errors, so we cannot say it loses clearly, but it does not win either.
- **What we did:** the main model stays on log. Switching to 1/0 because it scored better on the test would tune the model on the data it is judged on (R2), and the matrix is frozen. A fair next step is to test 1/0 on a validation split from the training period and adopt it if it still wins.